# 09. Kịch bản bổ sung KH: giá trị của thông tin kế hoạch xả Hủa Na ngày t+1

Thiết kế, giả thuyết và cách diễn giải được viết **trước khi chạy** ở `docs/ThietKe_KichBan_KeHoachXa.md`.

Mọi biến thể = E2 (F1) + Q ra Hủa Na ngày t+1, ghép như FUT ghép mưa tương lai:
- **KH_TD:** huấn luyện và dự báo bằng Q ra t+1 thực đo (cận trên);
- **KH_AD{σ}:** mô hình KH_TD, khi dự báo nhận kế hoạch có sai số σ (tình huống hiện nay: chưa có lưu trữ kế hoạch xả);
- **KH_SS{σ}:** huấn luyện và dự báo bằng kế hoạch có sai số σ.

Sai số kế hoạch: Q̃ = max(0, Q·(1 + σ·z)), z ~ N(0, 1) theo ngày; σ ∈ {10, 20, 30, 50 %}.

Quy trình:
- siêu tham số giữ như E2, hàm mất mát cơ bản;
- 6 mô hình (không có Persistence);
- CV 3 fold × 3 hạt giống, huấn luyện cuối 5 hạt giống;
- **không có bước lựa chọn** giữa các biến thể, tập kiểm tra chấm một lần.

Mã: `src/plan_scenario.py` (không sửa `src/data/`, dữ liệu đóng băng v1 vẫn được xác minh).

In [ ]:
# ── Google Colab: gắn Google Drive, vào thư mục dự án, cài đúng phiên bản thư viện. Chạy trên máy thì tự bỏ qua. ──
import os, sys, glob, subprocess
DU_AN = '/content/drive/MyDrive/T_DAThS_Ninh'   # ← thư mục dự án trên Google Drive (notebook 01, mục 0 tạo ra)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(os.path.join(DU_AN, 'src')):          # không thấy → tự tìm trong My Drive (sâu tối đa 3 cấp)
        hit = [h for k in range(1, 4) for h in glob.glob('/content/drive/MyDrive/' + '*/' * k + 'src/results_store.py')]
        assert hit, ('Chưa có dự án trên Google Drive: mở notebook 01_PhanTich_XuLy_DuLieu và chạy 2 ô của mục 0 '
                     '(lấy code từ GitHub, giải nén data/kết quả), hoặc sửa biến DU_AN ở đầu ô này')
        DU_AN = os.path.dirname(os.path.dirname(hit[0]))
    os.chdir(DU_AN)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)

import json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src' / 'results_store.py').exists():
    assert ROOT != ROOT.parent, 'Không tìm thấy thư mục dự án (thư mục chứa src/): mở notebook từ trong dự án'
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
from src.utils import load_config, results_dir, dataset_meta
cfg = load_config()
print('Thư mục dự án:', ROOT, '| kết quả lưu ở:', cfg['paths']['results'])
print('Thiết bị:', f'GPU {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else
      'CPU (trên Colab: Runtime → Change runtime type → GPU để chạy nhanh hơn)')
from src import pipeline, plan_scenario as K
from src import results_store as S
from src.evaluation import comparison as C
from src.evaluation.metrics import LABEL, DIGITS
from src.training.train import DISPLAY
from src.training.tuning import cv_summary
OUT = results_dir(cfg, 'ke_hoach_xa')
ds = pipeline.get_dataset(cfg, 1)
peaks, thr = pipeline.test_peaks(cfg, ds)
print('Đỉnh lũ test:', [d.date() for d in peaks], f'(ngưỡng Q99 train = {thr:.0f} m³/s)')
fi = ds.daily['hn_missing'].values; pos = ds.S['pos'].values
v_ = ds.view('FINAL', pipeline.feature_list(cfg, 'F1'), 3)
for nm, ix in [('train cuối', v_.idx_train), ('test', v_.idx_eval)]:
    print(f'{nm}: số mẫu có Q ra Hủa Na ngày t+1 thiếu (dùng giá trị gần nhất đã biết): {int(fi[pos[ix] + 1].sum())}')

## 1. Kiểm tra trước khi chạy: đường mã mới tái lập E2 đã lưu

Huấn luyện lại E2 (không ghép biến) qua đúng các hàm dùng cho kịch bản KH, hạt giống 0, so với dự báo test E2 đã lưu.

In [ ]:
rep = pd.DataFrame([{'mô hình': DISPLAY[m], 'chênh lớn nhất (m³/s)': K.check_e2_reproduction(cfg, ds, m)} for m in K.MODELS]).set_index('mô hình')
display(rep); rep.to_csv(OUT / 'tai_lap_E2.csv', encoding='utf-8-sig')
assert rep['chênh lớn nhất (m³/s)'].max() < 0.5

## 2. Chạy CV và huấn luyện cuối

In [ ]:
cvs, cvp, mets, preds = [], [], [], []
for m in K.MODELS:
    cv_m, cp_m = K.run_cv(cfg, ds, m)
    met_m, pr_m = K.run_final(cfg, ds, m, cv_m, peaks)
    cvs.append(cv_m); cvp.append(cp_m); mets.append(met_m); preds.append(pr_m)
    # ghi vào các file gộp, nhóm 'KH': results/ket_qua_thuc_nghiem.csv (chỉ tiêu), results/du_bao.csv.gz (dự báo)
    for gd, a, b in [('cv', cv_m, cp_m), ('test', met_m, pr_m)]:
        S.write_metrics(cfg, a, m, 1, gd, 'KH'); S.write_pred(cfg, b, m, 1, gd, 'KH')
cvk, cvpk, metk, predk = pd.concat(cvs), pd.concat(cvp), pd.concat(mets), pd.concat(preds)

## 3. Kết quả CV (NSE trung bình 3 fold), cùng E2 và FUT đã có

In [ ]:
cv0, met0, pred0 = C.load_all(cfg, 'H1')
EXPS = ['E2', 'FUT', 'KH_TD'] + [K.tag('AD', s) for s in K.SIGMAS] + [K.tag('SS', s) for s in K.SIGMAS]
cv_all = pd.concat([cv0[cv0.exp.isin(['E2', 'FUT']) & cv0.model.isin(K.MODELS)], cvk])
cvt = C.cv_matrix(cv_all)[EXPS]; display(cvt.round(3)); cvt.to_csv(OUT / 'cv_nse.csv', encoding='utf-8-sig')
cvq = C.cv_matrix(cv_all, 'NSE Q>Q95 CV')[EXPS]; print('NSE Q>Q95 (CV):'); display(cvq.round(3)); cvq.to_csv(OUT / 'cv_nse_q95.csv', encoding='utf-8-sig')

## 4. Kết quả trên tập kiểm tra (TB ± ĐLC giữa các hạt giống)

In [ ]:
met_all = pd.concat([met0[met0.exp.isin(['E2', 'FUT']) & met0.model.isin(K.MODELS)], metk])
KEYS = ['NSE', 'KGE', 'RMSE', 'MAE', 'PBIAS', 'NSE_Q95', 'RMSE_Q95', 'UNDER_Q95', 'Peak_RMSE', 'Peak_Bias', 'Peak_Timing']
tt = C.test_table(met_all, EXPS, KEYS); display(tt); tt.to_csv(OUT / 'test_chi_tieu.csv', encoding='utf-8-sig')
tn = met_all.groupby(['model', 'exp']).NSE.mean().unstack()[EXPS].rename(index=DISPLAY).loc[[DISPLAY[m] for m in K.MODELS]]
print('NSE test (TB hạt giống):'); display(tn.round(3)); tn.to_csv(OUT / 'test_nse.csv', encoding='utf-8-sig')

## 5. Bootstrap khối (tập kiểm tra, cùng một bộ chỉ số cho mọi cặp)

KTC 95 % không chứa 0 → có ý nghĩa (✔). P(A tốt hơn) = tỉ lệ lần bootstrap A tốt hơn B.

In [ ]:
pred_all = pd.concat([pred0[pred0.exp.isin(['E2', 'FUT']) & pred0.model.isin(K.MODELS) & (pred0.h == 1)], predk])
pairs = []
for m in K.MODELS:
    pairs += [(m, 'KH_TD', m, 'E2', f'{DISPLAY[m]}: KH_TD − E2'), (m, 'KH_TD', m, 'FUT', f'{DISPLAY[m]}: KH_TD − FUT')]
    pairs += [(m, K.tag(k, s), m, 'E2', f'{DISPLAY[m]}: {K.tag(k, s)} − E2') for k in ['AD', 'SS'] for s in K.SIGMAS]
pairs += [(m, 'KH_TD', 'ridge', 'KH_TD', f'{DISPLAY[m]} − Ridge-ARX (KH_TD)') for m in ['lstm', 'gru']]
pairs += [(m, 'E2', 'ridge', 'E2', f'{DISPLAY[m]} − Ridge-ARX (E2, đối chứng)') for m in ['lstm', 'gru']]
bk = C.bootstrap_multi(cfg, pred_all, pairs); display(bk); bk.to_csv(OUT / 'bootstrap.csv', encoding='utf-8-sig')

## 6. Mức sai số hoà vốn

σ lớn nhất mà biến thể còn hơn E2 có ý nghĩa về NSE (KTC 95 % > 0). "—" nghĩa là không mức nào có ý nghĩa.

In [ ]:
rows = []
for m in K.MODELS:
    r = {'mô hình': DISPLAY[m]}
    for k in ['AD', 'SS']:
        ok = [s for s in K.SIGMAS if bk.loc[f'{DISPLAY[m]}: {K.tag(k, s)} − E2', 'KTC NSE'].endswith('✔')
              and bk.loc[f'{DISPLAY[m]}: {K.tag(k, s)} − E2', 'ΔNSE'] > 0]
        r[f'KH_{k}: σ lớn nhất còn hơn E2 có ý nghĩa'] = f'{int(max(ok) * 100)} %' if ok else '—'
    r['KH_TD − E2 có ý nghĩa'] = '✔' if bk.loc[f'{DISPLAY[m]}: KH_TD − E2', 'KTC NSE'].endswith('✔') else ''
    rows.append(r)
hv = pd.DataFrame(rows).set_index('mô hình'); display(hv); hv.to_csv(OUT / 'muc_sai_so_hoa_von.csv', encoding='utf-8-sig')

## 7. Lợi ích theo mùa và trạng thái dòng chảy (MAE, mô tả, không kiểm định)

Như notebook 07, mục 12: nhóm theo **ngày đích**, ngưỡng Q50/Q90 từ 2019–2023, dự báo trung bình hạt giống. Hai nguồn: val
của CV (3 fold gộp) và test.

In [ ]:
daily_ = pd.read_csv(ROOT / cfg['paths']['processed'] / 'cuadat_huana_mua_daily.csv', parse_dates=['date'], index_col='date')
q_ = daily_.q_cd[(daily_.index <= '2023-12-31') & ~daily_.cd_missing]; Q50, Q90 = q_.quantile(.5), q_.quantile(.9)
def groups(dt, o):
    fl = dt.month.isin([7, 8, 9, 10])
    return {'cả năm': np.ones(len(o), bool), 'mùa lũ T7–T10': fl, 'mùa kiệt T11–T6': ~fl,
            f'Q thấp (< {Q50:.0f})': o < Q50, f'Q trung bình ({Q50:.0f}–{Q90:.0f})': (o >= Q50) & (o < Q90), f'Q cao (≥ {Q90:.0f})': o >= Q90}
cvp0 = S.read_pred(cfg, 'cv', 1, 'chinh', K.MODELS, 'E2')
G_EX = ['E2', 'KH_TD', 'KH_AD20', 'KH_SS20']
rows = []
for src_, data in [('val CV (3 fold)', pd.concat([cvp0, cvpk])), ('test', pred_all)]:
    for m in K.MODELS:
        for e in G_EX:
            d = data[(data.model == m) & (data.exp == e) & (data.h == 1) & data.dich_that.astype(bool)]
            keys = ['fold', 'ngay_goc'] if 'fold' in d and d['fold'].notna().all() else ['ngay_goc']
            a = d.groupby(keys).agg(o=('Q_do', 'first'), p=('Q_du_bao', 'mean')).reset_index()
            dt = pd.DatetimeIndex(a.ngay_goc) + pd.Timedelta(days=1)
            for g, mk in groups(dt, a.o.values).items():
                rows.append({'nguồn': src_, 'mô hình': DISPLAY[m], 'thực nghiệm': e, 'nhóm': g, 'n': int(mk.sum()),
                             'MAE': float(np.abs(a.p.values[mk] - a.o.values[mk]).mean())})
hs = pd.DataFrame(rows)
piv = hs.pivot_table(index=['nguồn', 'nhóm', 'mô hình'], columns='thực nghiệm', values='MAE', sort=False)[G_EX]
for e in G_EX[1:]: piv[f'Δ% {e}'] = 100 * (piv[e] / piv['E2'] - 1)
piv['n'] = hs[hs['thực nghiệm'] == 'E2'].set_index(['nguồn', 'nhóm', 'mô hình']).n
piv.to_csv(OUT / 'theo_mua_trang_thai.csv', encoding='utf-8-sig')
med = piv.groupby(level=['nguồn', 'nhóm'], sort=False)[[f'Δ% {e}' for e in G_EX[1:]]].median()
print('Δ% MAE so với E2, trung vị 6 mô hình:'); display(med.round(1)); med.to_csv(OUT / 'theo_mua_trang_thai_trung_vi.csv', encoding='utf-8-sig')

## 8. Đỉnh lũ trên tập kiểm tra (mô tả)

In [ ]:
pk = C.peak_table(pred_all, peaks, [(m, e) for m in ['ridge', 'lstm', 'gru'] for e in ['E2', 'KH_TD', 'KH_AD20']])
display(pk.round(1)); pk.to_csv(OUT / 'dinh_lu.csv', index=False, encoding='utf-8-sig')

## 9. Hình

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
xs = [0] + [int(s * 100) for s in K.SIGMAS]
cols = {'ridge': 'C2', 'extra_trees': 'C4', 'xgboost': 'C1', 'rnn': 'C5', 'lstm': 'C3', 'gru': 'C0'}
for ax, k, ttl in zip(axs, ['AD', 'SS'], ['(a) Huấn luyện bằng Q ra thực đo, dự báo bằng kế hoạch sai số σ (KH_AD)',
                                          '(b) Huấn luyện và dự báo bằng kế hoạch sai số σ (KH_SS)']):
    for m in K.MODELS:
        ys, lo, hi = [], [], []
        for e in ['KH_TD'] + [K.tag(k, s) for s in K.SIGMAS]:
            r = bk.loc[f'{DISPLAY[m]}: {e} − E2']
            ys.append(r['ΔNSE']); a_, b_ = r['KTC NSE'].strip(' ✔').strip('[]').split(';'); lo.append(float(a_)); hi.append(float(b_))
        ax.plot(xs, ys, marker='o', color=cols[m], label=DISPLAY[m])
        if m in ('ridge', 'lstm'): ax.fill_between(xs, lo, hi, color=cols[m], alpha=.12)
    ax.axhline(0, color='k', lw=.8); ax.set_xlabel('sai số kế hoạch σ (%)'); ax.set_title(ttl, loc='left', fontsize=9)
axs[0].set_ylabel('ΔNSE test so với E2'); axs[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / 'fig_delta_nse_theo_sai_so.png'); plt.show()

a, b = pd.Timestamp('2025-07-01'), pd.Timestamp('2025-10-31')
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, m in zip(axs, ['ridge', 'lstm']):
    for e, c in [('E2', 'C0'), ('KH_TD', 'C3'), ('KH_AD20', 'C1')]:
        O, M_, P, idx = C.arrays(pred_all, m, e); t = idx + pd.Timedelta(days=1)
        if e == 'E2': ax.plot(pd.Series(O[:, 0], index=t).loc[a:b], 'k', lw=1.3, label='quan trắc')
        ax.plot(pd.Series(np.mean(P, axis=0)[:, 0], index=t).loc[a:b], color=c, lw=1, label=e)
    ax.set_title(f'{DISPLAY[m]}: mùa lũ 2025, dự báo t+1 (TB hạt giống)', loc='left', fontsize=10); ax.set_ylabel('Q đến CĐ (m³/s)'); ax.legend(fontsize=8, ncol=4)
plt.tight_layout(); plt.savefig(OUT / 'fig_mua_lu_2025.png'); plt.show()